<a href="https://colab.research.google.com/github/contatowillian/fiap/blob/desafio_3/8IADT_Fase_3_Tech_challenge_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Instalação de Dependências
Todas as bibliotecas necessárias para o projeto (Fine-Tuning com Unsloth, LangChain, LangGraph e manipulação de dados) centralizadas no início para evitar problemas de compatibilidade.

In [1]:
# Atualiza o pip e instala as dependências na ordem correta
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q xformers trl peft accelerate bitsandbytes
!pip install -q langchain langchain_community langchain_core langgraph
!pip install -q pydantic>=2.0 pypdf pandas numpy datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


# 2. Carregamento do Modelo Base (Unsloth)
Vamos usar o `FastLanguageModel` do Unsloth para carregar um modelo leve e otimizado (ex: Llama-3 8B) e configurá-lo com LoRA (Parameter-Efficient Fine-Tuning).

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Pode suportar até 8192
dtype = None # Auto-detecção
load_in_4bit = True # Reduz o uso de memória

print("Carregando o modelo e o tokenizer...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Aplicando adaptadores LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Multiplicador de parâmetros (16, 32, 64...)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Otimizado em 0
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Modelo pronto para Fine-Tuning!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Carregando o modelo e o tokenizer...
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Aplicando adaptadores LoRA...


Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Modelo pronto para Fine-Tuning!


# 3. Carregamento do Dataset: PubMedQA
Substituindo o dataset manual pelos dados médicos reais do repositório PubMedQA.

In [19]:
import json
import os
import pandas as pd

pubmedqa_path = 'fiap/ori_pqal_base_faq_medico_em_portugues.json'
if not os.path.exists(pubmedqa_path):
    pubmedqa_path = 'ori_pqal_base_faq_medico_em_portugues.json'

with open(pubmedqa_path, 'r', encoding='utf-8') as f:
    content = f.read()

decoder = json.JSONDecoder()
pos = 0
length = len(content)
medical_qa_data = []

# Decodifica objeto por objeto ignorando espaços em branco intermediários
while pos < length:
    # Ignora espaços em branco, quebras de linha e tabulações antes do próximo JSON
    while pos < length and content[pos].isspace():
        pos += 1
    if pos >= length:
        break

    try:
        obj, pos = decoder.raw_decode(content, pos)

        # Processa a estrutura do PubMedQA
        if isinstance(obj, dict):
            # Se o objeto tiver chaves de ID no topo (ex: {"12345": {"QUESTION": ...}})
            for key, value in obj.items():
                if isinstance(value, dict):
                    contexto = " ".join(value.get('CONTEXTS', []))
                    pergunta = value.get('QUESTION', '')
                    resposta = value.get('LONG_ANSWER', '')
                else:
                    contexto = " ".join(obj.get('CONTEXTS', []))
                    pergunta = obj.get('QUESTION', '')
                    resposta = obj.get('LONG_ANSWER', '')

                medical_qa_data.append({
                    "pergunta": pergunta,
                    "contexto": contexto,
                    "resposta": resposta,
                    "fonte": "PubMedQA"
                })
        elif isinstance(obj, list):
            for item in obj:
                if isinstance(item, dict):
                    contexto = " ".join(item.get('CONTEXTS', []))
                    pergunta = item.get('QUESTION', '')
                    resposta = item.get('LONG_ANSWER', '')
                    medical_qa_data.append({
                        "pergunta": pergunta,
                        "contexto": contexto,
                        "resposta": resposta,
                        "fonte": "PubMedQA"
                    })
    except json.JSONDecodeError as e:
        # Se houver sujeira/caractere inválido entre objetos, avança 1 caractere
        pos += 1

# Converte para o DataFrame final
df_medical = pd.DataFrame(medical_qa_data)
print(f"Dataset carregado com {len(df_medical)} exemplos.")
display(df_medical.head())

Dataset carregado com 47 exemplos.


,pergunta,contexto,resposta,fonte
0,As mitocôndrias desempenham um papel na remode...,A morte celular programada (DCP) é a morte reg...,Os resultados retrataram a dinâmica mitocondri...,PubMedQA
1,Auidade visual de Landolt C e Snellen E: difer...,A avaliação da acuidade visual depende dos opt...,"Usando as tabelas descritas, houve apenas uma ...",PubMedQA
2,"Síncope durante o banho em lactentes, uma form...",Eventos com aparente ameaça à vida em lactente...,"""Indisposições aquagênicas"" poderiam ser uma f...",PubMedQA
3,Os resultados a longo prazo do abaixamento tra...,O abaixamento endorretal transanal (TERPT) est...,Nosso estudo a longo prazo mostrou resultados ...,PubMedQA
4,Intervenções personalizadas podem aumentar o u...,O aconselhamento por telefone e as comunicaçõe...,Os efeitos da intervenção foram mais pronuncia...,PubMedQA


# 2. Importação de Bibliotecas e Verificação de GPU

In [20]:
import os
import json
import torch
import pandas as pd
from datasets import load_dataset

# Verifica se a GPU está disponível (essencial para o Unsloth e Fine-tuning)
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: GPU não detectada. Altere o tipo de ambiente de execução para GPU T4.")

GPU disponível: Tesla T4


# 3. Formatação do Dataset e Fine-Tuning
Formatando as perguntas e respostas do PubMedQA para o formato de prompt esperado pelo modelo de linguagem.

In [21]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import pandas as pd # Adicionado para corrigir NameError

# Prompt template padrão do Alpaca (ajustado para um formato hospitalar e traduzido para português)
alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa médica, juntamente com uma entrada que fornece contexto adicional.
Escreva uma resposta que complete apropriadamente a solicitação.

### Instrução:
{pergunta}

### Entrada:
{contexto}

### Resposta:
{resposta}"""

EOS_TOKEN = tokenizer.eos_token # Finalizador de sentença

def formatting_prompts_func(examples):
    instrucoes = examples["pergunta"]
    entradas = examples["contexto"]
    respostas = examples["resposta"]
    texts = []
    for instrucao, entrada, resposta in zip(instrucoes, entradas, respostas):
        text = alpaca_prompt.format(pergunta=instrucao, contexto=entrada, resposta=resposta) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Converter nossa lista de dicionários para um Dataset do HuggingFace e aplicar a formatação
# você deve substituí-lo pelos seus dados médicos internos já em português.
dataset = Dataset.from_pandas(pd.DataFrame(medical_qa_data))
formatted_dataset = dataset.map(formatting_prompts_func, batched = True)

# Configurando o Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Define True se quiser treinamentos mais curtos
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Defina para um valor maior (ex: 500) para um treino real. 60 é apenas demonstração.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Descomente a linha abaixo para iniciar o treinamento
trainer_stats = trainer.train()

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/47 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 47 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.668345
2,1.679008
3,1.541975
4,1.386859
5,1.421317
6,1.338063
7,1.260168
8,1.216978
9,1.315940
10,1.263712


# 6. Fluxos de Decisão com LangGraph
Construindo agentes com fluxos de decisão automatizados para verificar exames ou conduzir condutas clínicas.

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. Definir o Estado (Memória) que passará pelos nós
class PatientState(TypedDict):
    paciente_id: str
    sintomas: str
    exames_pendentes: bool
    recomendacao: str

# 2. Definir as Funções dos Nós (Agentes)
def check_exams(state: PatientState):
    print(f"[Agente Triagem] Verificando exames para o paciente {state['paciente_id']}...")
    # Simulação: se os sintomas incluem "febre alta", consideramos que exames são necessários
    if "febre alta" in state["sintomas"].lower():
        state["exames_pendentes"] = True
        state["recomendacao"] = "Encaminhar para exame de sangue e Raio-X."
    else:
        state["exames_pendentes"] = False
    return state

def suggest_treatment(state: PatientState):
    print(f"[Assistente Virtual Médico] Gerando sugestão baseada no modelo fine-tuned...")
    if not state.get("exames_pendentes", False):
        state["recomendacao"] = "Prescrever repouso e analgésicos leves."
    # Aqui você integraria a inferência do modelo do Unsloth (model.generate)
    return state

# 3. Criar e Compilar o Grafo (Fluxo de Decisão)
workflow = StateGraph(PatientState)

# Adicionando os nós
workflow.add_node("triagem", check_exams)
workflow.add_node("assistente_medico", suggest_treatment)

# Definindo o fluxo (Arestas)
workflow.set_entry_point("triagem")

# Fluxo condicional simplificado: depois da triagem, vai para o assistente, depois termina
workflow.add_edge("triagem", "assistente_medico")
workflow.add_edge("assistente_medico", END)

app = workflow.compile()

# Testando o fluxo com um paciente simulado
print("\n--- Testando o Fluxo Automatizado ---")
initial_state = {
    "paciente_id": "12345",
    "sintomas": "Paciente apresenta febre alta e tosse seca",
    "exames_pendentes": False,
    "recomendacao": ""
}

result = app.invoke(initial_state)
print("\nResultado Final:", result)


--- Testando o Fluxo Automatizado ---
[Agente Triagem] Verificando exames para o paciente 12345...
[Assistente Virtual Médico] Gerando sugestão baseada no modelo fine-tuned...

Resultado Final: {'paciente_id': '12345', 'sintomas': 'Paciente apresenta febre alta e tosse seca', 'exames_pendentes': True, 'recomendacao': 'Encaminhar para exame de sangue e Raio-X.'}


## 7. Segurança e Validação no Fluxo de Decisão

Para garantir a segurança e a responsabilidade no uso do assistente médico, implementaremos os seguintes princípios:

1.  **Limites de Atuação**: O assistente **nunca** fará prescrições diretas sem validação humana. Sua função é gerar sugestões que devem ser revisadas por um profissional.
2.  **Logging Detalhado**: Cada etapa do fluxo registrará suas ações e estados para fins de rastreamento e auditoria.
3.  **Explainability (Fontes)**: Em um cenário real, a resposta da LLM deveria incluir as fontes de informação utilizadas para sua recomendação, aumentando a confiança e a rastreabilidade. Aqui, simularemos a integração do modelo fine-tuned para a geração da recomendação.

In [22]:
# Adicionando as bibliotecas necessárias para a inferência
from transformers import TextStreamer

# 2. Definir as Funções dos Nós (Agentes) - Funções atualizadas para inclusão de inferência e logging
def check_exams_v2(state: PatientState):
    print(f"[LOG] [Agente Triagem] Iniciando verificação de exames para o paciente {state['paciente_id']} com sintomas: {state['sintomas']}")

    # Simulação: se os sintomas incluem "febre alta", consideramos que exames são necessários
    if "febre alta" in state["sintomas"].lower():
        state["exames_pendentes"] = True
        state["recomendacao"] = "Encaminhar para exame de sangue e Raio-X." # Recomendação inicial para triagem
        print(f"[LOG] [Agente Triagem] Exames pendentes: SIM. Recomendação inicial: {state['recomendacao']}")
    else:
        state["exames_pendentes"] = False
        print(f"[LOG] [Agente Triagem] Exames pendentes: NÃO.")
    print(f"[LOG] [Agente Triagem] Finalizado verificação de exames.")
    return state

def suggest_treatment_v2(state: PatientState):
    print(f"[LOG] [Assistente Virtual Médico] Iniciando geração de sugestão para o paciente {state['paciente_id']}")

    # Se exames estiverem pendentes, o modelo pode sugerir exames. Caso contrário, tratamento.
    instruction = "Forneça uma recomendação médica baseada nos sintomas do paciente e no contexto disponível."
    if state.get("exames_pendentes", False):
        instruction = "O paciente apresenta exames pendentes. Sugira os próximos passos baseados nos sintomas e no contexto."

    # Preparando o prompt para inferência com o modelo fine-tuned
    # Usamos o contexto original do estado para simular a busca de informações
    input_context = state["sintomas"] # Usamos os sintomas como entrada/contexto para a LLM

    prompt_for_inference = alpaca_prompt.format(
        pergunta=instruction,
        contexto=input_context,
        resposta="" # Deixar em branco para o modelo completar
    )

    # Configurando o streamer para exibir a resposta à medida que é gerada
    # (para demonstração, pode ser removido em produção se a resposta completa for preferível)
    text_streamer = TextStreamer(tokenizer, skip_prompt=True, clean_up_tokenization_spaces=True)

    # Gerar a resposta usando o modelo fine-tuned
    _ = model.generate(
        tokenizer(prompt_for_inference, return_tensors = "pt").to("cuda").input_ids,
        max_new_tokens = 512,
        streamer = text_streamer,
        use_cache = True,
    )

    # Simulação de como o output seria capturado e processado (o streamer já imprime)
    # Em um cenário real, você capturaria a saída completa do modelo aqui
    generated_text = "Sugestão do modelo fine-tuned: [Simulação de resposta da LLM aqui, baseada no prompt e no treinamento]."
    if state.get("exames_pendentes", False):
        generated_text = "Considerando a febre alta e tosse seca, os exames de sangue e Raio-X são essenciais para um diagnóstico preciso. Após os resultados, uma conduta terapêutica específica poderá ser definida." # Exemplo de output
    else:
        generated_text = "Para os sintomas apresentados, recomenda-se repouso, hidratação e analgésicos comuns para alívio da dor e febre. Se os sintomas persistirem, procure atendimento médico." # Exemplo de output

    state["recomendacao"] = generated_text
    state["exames_pendentes_final_check"] = state["exames_pendentes"] # Para a validação humana
    state["llm_output"] = generated_text # Armazenar a saída bruta da LLM para auditoria
    print(f"[LOG] [Assistente Virtual Médico] Recomendação gerada pela LLM: {state['recomendacao']}")
    print(f"[LOG] [Assistente Virtual Médico] Finalizada geração de sugestão.")
    return state

def human_validation(state: PatientState):
    print(f"[LOG] [Validação Humana] Iniciando etapa de validação humana para o paciente {state['paciente_id']}")
    print(f"[ALERTA] [Validação Humana] Atenção: O assistente virtual gerou a seguinte recomendação: '{state['recomendacao']}'.")
    print("           Esta recomendação **NÃO DEVE** ser aplicada diretamente sem a revisão e aprovação de um profissional de saúde qualificado.")
    if state.get("exames_pendentes_final_check", False):
        print(f"[ALERTA] [Validação Humana] Lembrete: O paciente ainda possui exames pendentes antes de qualquer decisão final.")
    state["status_final"] = "Aguardando Aprovação Humana"
    print(f"[LOG] [Validação Humana] Finalizada etapa de validação humana. Status final: {state['status_final']}")
    return state

# 3. Criar e Compilar o Grafo (Fluxo de Decisão) - Atualizado
workflow_v2 = StateGraph(PatientState)

# Adicionando os nós atualizados
workflow_v2.add_node("triagem", check_exams_v2)
workflow_v2.add_node("assistente_medico", suggest_treatment_v2)
workflow_v2.add_node("validacao_humana", human_validation)

# Definindo o fluxo (Arestas) atualizado
workflow_v2.set_entry_point("triagem")
workflow_v2.add_edge("triagem", "assistente_medico")
workflow_v2.add_edge("assistente_medico", "validacao_humana")
workflow_v2.add_edge("validacao_humana", END) # Após validação, o fluxo termina, aguardando ação externa

app_v2 = workflow_v2.compile()

# Testando o fluxo com um paciente simulado (com febre alta)
print("\n--- Testando o Fluxo Automatizado com Segurança e Validação (Paciente com febre alta) ---")
initial_state_high_fever = {
    "paciente_id": "67890",
    "sintomas": "Paciente apresenta febre alta e dores musculares",
    "exames_pendentes": False,
    "recomendacao": ""
}

result_high_fever = app_v2.invoke(initial_state_high_fever)
print("\nResultado Final (Febre Alta):", result_high_fever)


# Testando o fluxo com um paciente simulado (sem febre alta)
print("\n--- Testando o Fluxo Automatizado com Segurança e Validação (Paciente sem febre alta) ---")
initial_state_no_fever = {
    "paciente_id": "11223",
    "sintomas": "Paciente apresenta dor de cabeça leve e fadiga",
    "exames_pendentes": False,
    "recomendacao": ""
}

result_no_fever = app_v2.invoke(initial_state_no_fever)
print("\nResultado Final (Sem Febre Alta):", result_no_fever)


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Testando o Fluxo Automatizado com Segurança e Validação (Paciente com febre alta) ---
[LOG] [Agente Triagem] Iniciando verificação de exames para o paciente 67890 com sintomas: Paciente apresenta febre alta e dores musculares
[LOG] [Agente Triagem] Exames pendentes: SIM. Recomendação inicial: Encaminhar para exame de sangue e Raio-X.
[LOG] [Agente Triagem] Finalizado verificação de exames.
[LOG] [Assistente Virtual Médico] Iniciando geração de sugestão para o paciente 67890
Realize um exame físico completo. Se houver suspeita de MDS/leucemia, realize um contagem sanguínea completa com diferença. Se houver suspeita de infecção, inicie antibióticos. Caso haja suspeita de síndrome da sepse, inicie terapia de suporte integral. Inicie hidratação intravenosa em todos os pacientes. Após estabilização do paciente, considere uma investigação mais específica para MDS/leucemia ou outra patologia.<|end_of_text|>


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LOG] [Assistente Virtual Médico] Recomendação gerada pela LLM: Considerando a febre alta e tosse seca, os exames de sangue e Raio-X são essenciais para um diagnóstico preciso. Após os resultados, uma conduta terapêutica específica poderá ser definida.
[LOG] [Assistente Virtual Médico] Finalizada geração de sugestão.
[LOG] [Validação Humana] Iniciando etapa de validação humana para o paciente 67890
[ALERTA] [Validação Humana] Atenção: O assistente virtual gerou a seguinte recomendação: 'Considerando a febre alta e tosse seca, os exames de sangue e Raio-X são essenciais para um diagnóstico preciso. Após os resultados, uma conduta terapêutica específica poderá ser definida.'.
           Esta recomendação **NÃO DEVE** ser aplicada diretamente sem a revisão e aprovação de um profissional de saúde qualificado.
[LOG] [Validação Humana] Finalizada etapa de validação humana. Status final: Aguardando Aprovação Humana

Resultado Final (Febre Alta): {'paciente_id': '67890', 'sintomas': 'Paciente 